In [ ]:
import numpy as np
import time

# ── 1. 데이터 ───────────────────────────────────────────────────
CROPS    = ['양상추','토마토','대파','오이','고추','시금치','무','배추','옥수수']
N_CROP, N_PERIOD = 9, 24
area_per = np.array([0.10,0.30,0.15,0.40,0.25,0.05,0.20,0.20,0.20])
grow_O   = np.array([4,8,6,5,10,3,6,5,6])
grow_H   = np.array([3,6,5,4,8,2,5,4,5])

price_O_monthly = np.array([
    [2500,2500,2500,2500,2500,0,0,0,0,0,2200,2200],
    [0,0,0,0,15000,18000,18000,15000,12000,0,0,0],
    [0,0,4000,4000,4500,5000,5000,5500,5000,4500,0,0],
    [0,0,0,0,11000,11000,13000,15000,13000,11000,0,0],
    [0,0,0,0,0,0,10000,12000,12000,10000,0,0],
    [800,800,600,600,0,0,0,0,700,700,800,800],
    [0,0,0,0,0,0,0,0,4500,4800,4800,4800],
    [0,0,0,0,0,0,0,0,3200,3200,3600,3600],
    [0,0,0,0,0,0,3000,3500,3000,0,0,0],
])
price_H_monthly = np.array([
    [3500,3500,3200,3200,3200,3000,3000,3500,3500,3200,3200,3200],
    [0,0,25000,25000,21000,25000,25000,21000,18000,20000,22000,22000],
    [6000,6000,6000,6000,6500,7000,7000,8000,7000,6500,7000,7000],
    [0,0,15000,15000,15000,16000,18000,21000,18000,16000,18000,18000],
    [0,0,15000,15000,15000,15000,14000,17000,17000,15000,18000,18000],
    [1200,1200,1000,1000,1000,1000,1000,1200,1000,1000,1200,1200],
    [6500,6500,6000,6000,6000,6000,6000,6500,6000,6500,6500,6500],
    [5500,5500,5000,5000,5000,5000,5000,5500,4800,4800,5000,5000],
    [0,0,0,0,4500,4500,4200,5000,4200,4500,0,0],
])

AREA_O, AREA_H = 1200.0, 300.0
MIN_O, MAX_O   = AREA_O * 0.05, AREA_O * 0.25
MIN_H, MAX_H   = AREA_H * 0.05, AREA_H * 0.25

valid_O = np.zeros((N_CROP, N_PERIOD), dtype=bool)
valid_H = np.zeros((N_CROP, N_PERIOD), dtype=bool)
rev_O   = np.zeros((N_CROP, N_PERIOD))
rev_H   = np.zeros((N_CROP, N_PERIOD))
for i in range(N_CROP):
    for t in range(N_PERIOD):
        ht = t + grow_O[i] - 1
        if ht < N_PERIOD and price_O_monthly[i, ht//2] > 0:
            valid_O[i,t] = True
            rev_O[i,t]   = price_O_monthly[i, ht//2] / area_per[i]
        ht = t + grow_H[i] - 1
        if ht < N_PERIOD and price_H_monthly[i, ht//2] > 0:
            valid_H[i,t] = True
            rev_H[i,t]   = price_H_monthly[i, ht//2] / area_per[i]

vars_O = [(i,t) for i in range(N_CROP) for t in range(N_PERIOD) if valid_O[i,t]]
vars_H = [(i,t) for i in range(N_CROP) for t in range(N_PERIOD) if valid_H[i,t]]
nO, nH = len(vars_O), len(vars_H)
n = nO + nH

In [ ]:
# ── 2. LP 제약 행렬 ─────────────────────────────────────────────
# 변수: [xO(nO개) | xH(nH개)]  (연속, 0 <= x <= MAX)
# 제약: 반기별 노지 용량(24) + 반기별 하우스 용량(24) + 상한(n)
def build_lp():
    c = np.array([rev_O[i,t] for i,t in vars_O] +
                 [rev_H[i,t] for i,t in vars_H])
    n_con = N_PERIOD * 2 + n
    A = np.zeros((n_con, n))
    b = np.zeros(n_con)
    for p in range(N_PERIOD):
        for k,(i,t) in enumerate(vars_O):
            if t <= p < t + grow_O[i]: A[p, k] = 1
        b[p] = AREA_O
        for k,(i,t) in enumerate(vars_H):
            if t <= p < t + grow_H[i]: A[N_PERIOD+p, nO+k] = 1
        b[N_PERIOD+p] = AREA_H
    for k in range(nO): A[2*N_PERIOD+k, k] = 1;       b[2*N_PERIOD+k] = MAX_O
    for k in range(nH): A[2*N_PERIOD+nO+k, nO+k] = 1; b[2*N_PERIOD+nO+k] = MAX_H
    return c, A, b

c_lp, A_lp, b_lp = build_lp()

# ── 3. Simplex (Tableau 방식) ───────────────────────────────────
# max c^T x  s.t. Ax <= b, x >= 0
# slack 추가 → 초기 BFS: slack = b
def simplex(c, A, b):
    m, nv = A.shape
    T = np.zeros((m+1, nv+m+1))
    T[:m, :nv]     = A
    T[:m, nv:nv+m] = np.eye(m)
    T[:m, -1]       = b
    T[m, :nv]       = -c
    basis = list(range(nv, nv+m))
    for _ in range(50000):
        col = int(np.argmin(T[m, :-1]))
        if T[m, col] >= -1e-9: break
        ratios = [(T[r,-1]/T[r,col], r) for r in range(m) if T[r,col] > 1e-9]
        if not ratios: return None, None
        _, row = min(ratios)
        basis[row] = col
        T[row] /= T[row, col]
        for r in range(m+1):
            if r != row: T[r] -= T[r, col] * T[row]
    x = np.zeros(nv+m)
    for r, bi in enumerate(basis): x[bi] = T[r, -1]
    return -T[m, -1], x[:nv]

# ── 4. Branch & Bound ───────────────────────────────────────────
# MILP 조건: x = 0 이거나 x >= MIN (심거나 안 심거나)
# Branch: LP 해에서 0 < x[k] < MIN 인 변수 발견 시
#   - Branch 0: x[k] = 0  (hi[k] = 0)
#   - Branch 1: x[k] >= MIN  (lo[k] = MIN)
def branch_and_bound():
    min_bound = np.array([MIN_O]*nO + [MIN_H]*nH)
    hi0       = np.array([MAX_O]*nO + [MAX_H]*nH)

    best_val = -np.inf
    best_x   = None
    nodes    = 0
    stack    = [(np.zeros(n), hi0)]

    while stack:
        lo, hi = stack.pop()
        nodes += 1

        b2 = b_lp.copy()
        for k in range(nO): b2[2*N_PERIOD+k]    = hi[k]
        for k in range(nH): b2[2*N_PERIOD+nO+k] = hi[nO+k]

        b_adj = b2 - A_lp @ lo
        if np.any(b_adj < -1e-9): continue

        val, x_rel = simplex(c_lp, A_lp, b_adj)
        if val is None: continue

        x_rel = np.clip(x_rel + lo, lo, hi)
        val   = float(c_lp @ x_rel)

        if val <= best_val + 1e-6: continue  # pruning

        frac_k = next((k for k in range(n)
                       if 1e-6 < x_rel[k] < min_bound[k] - 1e-6), -1)

        if frac_k == -1:
            if val > best_val:
                best_val, best_x = val, x_rel.copy()
        else:
            hi1 = hi.copy(); hi1[frac_k] = 0.0
            stack.append((lo.copy(), hi1))
            lo2 = lo.copy(); lo2[frac_k] = min_bound[frac_k]
            stack.append((lo2, hi.copy()))

    return best_val, best_x, nodes

# ── 5. 실행 ─────────────────────────────────────────────────────
t0 = time.perf_counter()
best_val, best_x, nodes = branch_and_bound()
elapsed = time.perf_counter() - t0

xO = np.zeros((N_CROP, N_PERIOD))
xH = np.zeros((N_CROP, N_PERIOD))
for k,(i,t) in enumerate(vars_O): xO[i,t] = best_x[k]
for k,(i,t) in enumerate(vars_H): xH[i,t] = best_x[nO+k]

crop_rev  = np.array([(rev_O[i]*xO[i]+rev_H[i]*xH[i]).sum() for i in range(N_CROP)])
total_rev = crop_rev.sum()

# ── 6. 출력 ─────────────────────────────────────────────────────
print("=" * 52)
print(f"  MILP (LP + B&B): {total_rev/1e6:.4f} 백만원")
print(f"  탐색 노드: {nodes}개  /  소요 시간: {elapsed*1000:.1f} ms")
print("=" * 52)
print(f"\n{'작물':<6} {'수익(백만원)':>12} {'노지 파종합(m2)':>14} {'하우스 파종합(m2)':>16}")
print("-" * 52)
for i in range(N_CROP):
    print(f"{CROPS[i]:<6} {crop_rev[i]/1e6:>12.3f}"
          f" {xO[i].sum():>14.1f} {xH[i].sum():>16.1f}")
print("-" * 52)
print(f"{'합계':<6} {total_rev/1e6:>12.4f} {xO.sum():>14.1f} {xH.sum():>16.1f}")

print("\n[제약 확인]")
occ_O = np.zeros(N_PERIOD); occ_H = np.zeros(N_PERIOD)
for i in range(N_CROP):
    for t in range(N_PERIOD):
        if xO[i,t] > 0:
            for p in range(t, t+grow_O[i]): occ_O[p] += xO[i,t]
        if xH[i,t] > 0:
            for p in range(t, t+grow_H[i]): occ_H[p] += xH[i,t]
print(f"노지   최대 점유: {occ_O.max():.1f} / {AREA_O:.1f} m2")
print(f"하우스 최대 점유: {occ_H.max():.1f} / {AREA_H:.1f} m2")
if occ_O.max() <= AREA_O+1e-6 and occ_H.max() <= AREA_H+1e-6:
    print("모든 반기에서 면적 제약 만족.")

In [ ]:
# ── 3. Simplex (Tableau 방식) ───────────────────────────────────
# max c^T x  s.t. Ax <= b, x >= 0
# slack 추가 → Ax + s = b, 초기 BFS: s = b

def simplex(c, A, b):
    m, nv = A.shape
    # Tableau: [A | I | b]  목적행: [-c | 0 | 0]
    T = np.zeros((m+1, nv+m+1))
    T[:m, :nv]      = A
    T[:m, nv:nv+m]  = np.eye(m)
    T[:m, -1]        = b
    T[m, :nv]        = -c   # 최소화 형식으로 저장 (부호 반전)

    basis = list(range(nv, nv+m))

    for _ in range(50000):
        # 피벗 열: 목적행에서 가장 음수인 열
        col = int(np.argmin(T[m, :-1]))
        if T[m, col] >= -1e-9:
            break  # 최적

        # 피벗 행: 최소 비율 테스트
        ratios = [(T[r,-1]/T[r,col], r) for r in range(m) if T[r,col] > 1e-9]
        if not ratios:
            return None, None   # Unbounded
        _, row = min(ratios)

        basis[row] = col
        T[row] /= T[row, col]
        for r in range(m+1):
            if r != row:
                T[r] -= T[r, col] * T[row]

    x = np.zeros(nv + m)
    for r, bi in enumerate(basis):
        x[bi] = T[r, -1]

    return -T[m, -1], x[:nv]


# 테스트
lp_val, lp_x = simplex(c_lp, A_lp, b_lp)
print(f"LP relaxation 최적값: {lp_val/1e6:.4f} 백만원  (MILP의 상한)")